In [6]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
!unzip -q "/content/drive/MyDrive/KhPI_HW/linear_algebra_intro_practice-main.zip" -d /content/idpp
!find /content/idpp -name "matrix_mapping.py"
!find /content/idpp -name "pyproject.toml"

replace /content/idpp/linear_algebra_intro_practice-main/.gitignore? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
/content/idpp/linear_algebra_intro_practice-main/matrix_mapping.py
/content/idpp/linear_algebra_intro_practice-main/.venv/lib/python3.13/site-packages/jupyterlab/tests/mock_packages/test-hyphens/pyproject.toml
/content/idpp/linear_algebra_intro_practice-main/.venv/lib/python3.13/site-packages/jupyterlab/tests/mock_packages/test_no_hyphens/pyproject.toml
/content/idpp/linear_algebra_intro_practice-main/.venv/lib/python3.13/site-packages/jupyterlab/tests/mock_packages/test-hyphens-underscore/pyproject.toml
/content/idpp/linear_algebra_intro_practice-main/pyproject.toml


In [8]:
%cd /content/idpp/linear_algebra_intro_practice-main

/content/idpp/linear_algebra_intro_practice-main


In [9]:
!pip -q install uv
!uv sync
!uv run pytest tests/test_matrix_mapping.py -v

Resolved 113 packages in 4ms
Checked 109 packages in 2ms
============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-9.1.1, pluggy-1.6.0 -- /content/idpp/linear_algebra_intro_practice-main/.venv/bin/python
cachedir: .pytest_cache
rootdir: /content/idpp/linear_algebra_intro_practice-main
configfile: pyproject.toml
plugins: platformdirs-4.13.0, anyio-4.15.1
collected 0 items                                                              

============================ no tests ran in 0.02s =============================
ERROR: file or directory not found: tests/test_matrix_mapping.py



In [11]:
# Перезавантажуємо модуль, щоб Colab точно взяв останню версію matrix_mapping.py
import importlib
import numpy as np
import matrix_mapping

importlib.reload(matrix_mapping)


def test_negative_matrix_basic_and_immutability():
    # Перевіряємо інвертування знаків елементів та те, що вихідний масив не змінюється
    x = np.array([[1.0, -2.0], [3.0, 0.0]])
    x_before = x.copy()

    result = matrix_mapping.negative_matrix(x)

    np.testing.assert_allclose(result, [[-1.0, 2.0], [-3.0, 0.0]])
    np.testing.assert_allclose(x, x_before), "Функція не повинна змінювати вхідний масив (mutability)"


def test_reverse_matrix_vector_and_matrix():
    # Перевіряємо розворот одновимірного вектора та двовимірної матриці
    v = np.array([1.0, 2.0, 3.0])
    np.testing.assert_allclose(matrix_mapping.reverse_matrix(v), [3.0, 2.0, 1.0])

    m = np.array([[1.0, 2.0], [3.0, 4.0]])
    # np.flip на двовимірному масиві розгортає вздовж усіх осей
    np.testing.assert_allclose(matrix_mapping.reverse_matrix(m), [[4.0, 3.0], [2.0, 1.0]])


def test_affine_transform_identity():
    # Перевіряємо нейтральне перетворення (нульовий кут, масштаб 1, без зсуву та перенесення)
    x = np.array([[1.0, 2.0], [3.0, 4.0]])

    result = matrix_mapping.affine_transform(
        x, alpha_deg=0.0, scale=(1.0, 1.0), shear=(0.0, 0.0), translate=(0.0, 0.0)
    )

    np.testing.assert_allclose(result, x)


def test_affine_transform_translation_vector():
    # Перевіряємо перенесення для вектор-стовпця (розмірність n*1)
    x = np.array([[0.0], [0.0]])

    # Зсув на вектор (5.0, -3.0) при нейтральному повороті та масштабі
    result = matrix_mapping.affine_transform(
        x, alpha_deg=0.0, scale=(1.0, 1.0), shear=(0.0, 0.0), translate=(5.0, -3.0)
    )

    np.testing.assert_allclose(result, [[5.0], [-3.0]])


def test_affine_transform_matrix_grid():
    # Перевіряємо роботу гілки для матриці n*n (перерахунок вузлів сітки)
    # Створимо невелику матрицю 3x3
    x = np.array([
        [1.0, 2.0, 3.0],
        [4.0, 5.0, 6.0],
        [7.0, 8.0, 9.0]
    ])

    # Виконуємо перенесення сітки
    result = matrix_mapping.affine_transform(
        x, alpha_deg=0.0, scale=(1.0, 1.0), shear=(0.0, 0.0), translate=(0.0, 0.0)
    )

    # При нульовому перенесенні матриця має залишитися колишн.
    np.testing.assert_allclose(result, x, atol=1e-7)


def test_affine_transform_immutability():
    # Перевіряємо, що аффінне перетворення не змінює переданий об'єкт в пам'яті
    x = np.array([[1.0, 2.0], [3.0, 4.0]])
    x_before = x.copy()

    _ = matrix_mapping.affine_transform(
        x, alpha_deg=45.0, scale=(2.0, 2.0), shear=(0.1, 0.1), translate=(1.0, 1.0)
    )

    np.testing.assert_allclose(x, x_before)


# Список усіх тестів для запуску
tests = [
    test_negative_matrix_basic_and_immutability,
    test_reverse_matrix_vector_and_matrix,
    test_affine_transform_identity,
    test_affine_transform_translation_vector,
    test_affine_transform_matrix_grid,
    test_affine_transform_immutability,
]

failed = []

for test in tests:
    try:
        test()
        print(f"PASS: {test.__name__}")
    except Exception as error:
        failed.append((test.__name__, str(error)))
        print(f"FAIL: {test.__name__}\n  {error}")

if failed:
    raise AssertionError(f"\nНе пройдено тестів: {len(failed)}")

print(f"\nУсі {len(tests)} локальних тестів для матричних перетворень пройдено.")

PASS: test_negative_matrix_basic_and_immutability
PASS: test_reverse_matrix_vector_and_matrix
PASS: test_affine_transform_identity
PASS: test_affine_transform_translation_vector
PASS: test_affine_transform_matrix_grid
PASS: test_affine_transform_immutability

Усі 6 локальних тестів для матричних перетворень пройдено.


# Код для matrix_mapping.pt

In [ ]:
import numpy as np

def negative_matrix(x: np.ndarray) -> np.ndarray:
    """
    Returns the negation of each element in the input vector or matrix.

    Args:
        x (np.ndarray): A vector (n*1) or matrix (n*n).

    Returns:
        np.ndarray: A matrix with each element negated.
    """
    #raise NotImplementedError
    return -x

def reverse_matrix(x: np.ndarray) -> np.ndarray:
    """
    Returns the input vector or matrix with the order of elements reversed.

    Args:
        x (np.ndarray): A vector (n*1) or matrix (n*n).

    Returns:
        np.ndarray: A matrix with the order of elements reversed.
    """
    #raise NotImplementedError
    # np.flip -- виконуємо розворот без зміни самих бітів float
    # .copy() -- створюємо незалежний, безперервний у пам'яті масив
    # захищає від пасток зміни вихідного масиву та проблем з C-contiguous
    return np.flip(x).copy()

def affine_transform(
    x: np.ndarray, alpha_deg: float, scale: tuple[float, float], shear: tuple[float, float],
    translate: tuple[float, float],
) -> np.ndarray:
    """Compute affine transformation

    Args:
        x (np.ndarray): vector n*1 or matrix n*n.
        alpha_deg (float): rotation angle in deg.
        scale (tuple[float, float]): x, y scale factor.
        shear (tuple[float, float]): x, y shear factor.
        translate (tuple[float, float]): x, y translation factor.

    Returns:
        np.ndarray: transformed matrix.
    """
    #raise NotImplementedError
    # матриця трансформац. 2х2 та вектор сдвигу
    sx, sy = scale
    shx, shy = shear
    theta = np.deg2rad(alpha_deg)
    c, s = np.cos(theta), np.sin(theta)

    A = np.array([[c, -s], [s, c]]) @ np.array([[1.0, shx], [shy, 1.0]]) @ np.array([[sx, 0.0], [0.0, sy]])
    t = np.array(translate).reshape(2, 1)

    # Якщо на вхід вектор n*1
    if x.shape[1] == 1 and x.shape[0] <= 3:
        return (A @ x[:2, :].astype(float)) + t

    # Якщо на вхід матриця n*n — перерахув. вузли сітки
    n = x.shape[0]
    res = np.zeros_like(x, dtype=float)

    # Обернена матриця для відображ. без пропуск.
    A_inv = np.linalg.inv(A)

    # сітка з індекс.
    y, x_idx = np.indices((n, n))
    coords = np.stack([x_idx.ravel(), y.ravel()])

    # Перенос. вузли до початков. системи
    old_coords = A_inv @ (coords - t)
    old_x = np.round(old_coords[0]).astype(int).reshape(n, n)
    old_y = np.round(old_coords[1]).astype(int).reshape(n, n)

    # щоб не вийти за межі розрахув. області
    mask = (old_x >= 0) & (old_x < n) & (old_y >= 0) & (old_y < n)

    # Результ. значення вузлів (за маскою залиш. нулі)
    res[y[mask], x_idx[mask]] = x[old_y[mask], x0_c_not_needed_here := old_x[mask]]

    return res